In [ ]:
import os, re, numpy as np
import pandas as pd

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score, precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import matthews_corrcoef
from xgboost import XGBClassifier
from sklearn.svm import SVC
import os

In [ ]:
# =========================================================
# CONSTANTS: FS, EPOCH LENGTH, SOP, SPH, MODELS, OUTPUT_DIR
# =========================================================
# Note: Here, SOP is used as the interval in which the seizure is expected to occur after the SPH, which is the time after an alarm that no new alarm is confirmed

FS = 256
SOP=1800
SPH=900
EPOCH_LEN = 10
models = {'Random Forest': RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=4, max_features="sqrt", random_state=42, n_jobs=-1),
         'XGBoost':XGBClassifier(n_estimators=300, max_depth=10, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective='binary:logistic',eval_metric='logloss', random_state=42),
         'SVM':SVC(kernel='linear',C=100,gamma = 'scale',random_state=42)}
save_dir = '/kaggle/working/Thres0.6'

In [ ]:
# =========================================================
# FUNTIONS FOR TRAINING AND LEAVE-ONE-SEIZURE-OUT STRATEGY
# =========================================================
def pivot_df(df_, index, value_cols, columns='Band'):
    df_pivot = df_.pivot(index=index, columns=columns, values=value_cols)
    
    df_pivot.columns = [f'{col}_{band}' for col, band in df_pivot.columns]

    final_df = df_pivot.reset_index()
    return final_df

def labeling_for_test(df_seizure, sop=SOP, sph=SPH):
    
    n_epochs = len(df_seizure)
    epoch_nearest_idx = n_epochs - (sph/EPOCH_LEN)
    epoch_farest_idx = n_epochs - (sop/EPOCH_LEN+sph/EPOCH_LEN)
    
    df_seizure.loc[(df_seizure['Epoch'] <= epoch_nearest_idx) & (df_seizure['Epoch'] >= epoch_farest_idx), 'Class'] = int(1)
    df_seizure.loc[(df_seizure['Epoch'] >= epoch_nearest_idx) | (df_seizure['Epoch'] <= epoch_farest_idx), 'Class'] = int(0)
    df_seizure['Class'] = df_seizure['Class'].astype(int)
    return df_seizure

def calculate_FAR(y_pred, y_test, k=90, refractory=SPH+SOP):
    false_alarms = 0
    idx=0
    total_refractory_epochs = 0
    while idx<len(y_pred):
        if y_test[idx]==1:
            break
        if y_pred[idx]==1:
            if np.all(y_pred[idx:idx+k]==1):
                if np.all(y_test[idx:idx+k]==0):
                    false_alarms += 1
                refractory_period = y_test[idx+k:idx+k+int(refractory/EPOCH_LEN)]
                refractory_epochs = np.sum(refractory_period == 0)
                total_refractory_epochs += refractory_epochs
                idx += k+int(refractory/EPOCH_LEN)
            else:
                idx+=1
        else:
            idx+=1
    num_non_preictal_epochs = np.sum(y_test[:idx] == 0)
    seconds = (num_non_preictal_epochs-total_refractory_epochs)*EPOCH_LEN
    hours = seconds/3600
    
    if hours <= 0:
        print('Invalid hours')
        return None
    return false_alarms/hours
            
def remove_duplicate_columns(df, keys=['Density', 'Characteristic_Path_Length', 'Weighted_Global_Efficiency']):
    cols_to_check = []

    for key in keys:
        cols_to_check.extend([c for c in df.columns if key in c])

    duplicate_cols = []
    kept_cols = {}

    for i, col1 in enumerate(cols_to_check):
        if col1 in duplicate_cols:
            continue

        kept_cols[col1] = None

        for col2 in cols_to_check[i+1:]:
            if col2 not in duplicate_cols and df[col1].equals(df[col2]):
                duplicate_cols.append(col2)

    rename_dict = {}
    for col in kept_cols:
        for key in keys:
            if key in col:
                rename_dict[col] = key
                break

    df = df.drop(columns=duplicate_cols)
    df = df.rename(columns=rename_dict)

    return df

def check_nan(df):
    if df.isna().any(axis=1).mean()<=0.05:
        return df.dropna()
    else:
        nan_ratio = df.isna().mean()

        cols_nan_gt5 = nan_ratio[nan_ratio > 0.05].index

        df = df.drop(columns=cols_nan_gt5)
        
        return df.dropna()

def balanced_accuracy(sensitivity, specificity):
    return (sensitivity+specificity)/2

def Evaluate_CV(preictal_df, interictal_df, df_test_pivoted, model,k, scaler=None):
    inter_indices = interictal_df['Epoch'].unique()
    ratio = len(preictal_df)/len(interictal_df['Epoch'].unique())
    seeds = list(range(len(preictal_df['SeizureID'].unique())))
    grouped = preictal_df.groupby("SeizureID")
    n_sz = grouped.ngroups
    n_sz_ = 0
    accuracy, specificity, sensitivity, f1, balanced, mcc, far = 0, 0, 0, 0, 0, 0, 0
    for i, (_, preictal_test) in enumerate(grouped):

        sid = preictal_test['SeizureID'].unique()[0]
        
        df_test_ = df_test_pivoted[df_test_pivoted['SeizureID']==sid]
        df_test_far = labeling_for_test(df_test_)
        
        y_test_far = df_test_far['Class'].values
        X_test_far = df_test_far.drop(columns=['SeizureID', 'Class', 'Epoch'])
        
        preictal_train = preictal_df.loc[preictal_df['SeizureID']!=preictal_test['SeizureID'].unique()[0]]
        
        n_pre = len(preictal_test['Epoch'].unique())
        n_inter = int(n_pre/ratio)
        n_pre_train = len(preictal_train)
        
        rng = np.random.default_rng(seeds[i])
        rng.shuffle(inter_indices)
        interictal_test = interictal_df.loc[interictal_df['Epoch'].isin(inter_indices[:n_inter])]
        interictal_train = interictal_df.loc[interictal_df['Epoch'].isin(inter_indices[n_inter:n_pre_train+n_inter])].copy()
        
        preictal_train = preictal_train.drop(columns=['SeizureID'])
        preictal_test = preictal_test.drop(columns=['SeizureID'])
        
        train_data = pd.concat([preictal_train, interictal_train], axis=0)
        test_data = pd.concat([preictal_test, interictal_test], axis=0)

        train_data['Task'] = 'Training'
        test_data['Task'] = 'Testing'

        df = pd.concat([train_data, test_data], axis=0)

        df = check_nan(df)

        train_data = df.loc[df['Task']=='Training'].drop(columns=['Task'])
        test_data = df.loc[df['Task']=='Testing'].drop(columns=['Task'])
        
        # Model training
        X_train = train_data.drop(columns=['Epoch', 'Class'])
        y_train = train_data['Class']

        X_test = test_data.drop(columns=['Epoch', 'Class'])
        y_test = test_data['Class']

        selector = SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, random_state=42),k=20)

        if scaler is not None:
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
            X_test_far = scaler.transform(X_test_far)
        
        X_train_selected = selector.fit_transform(X_train, y_train)
        
        X_test_selected = selector.transform(X_test)

        X_test_far_selected = selector.transform(X_test_far)

        model.fit(X_train_selected, y_train)
        y_pred = model.predict(X_test_selected)
        y_pred_far = model.predict(X_test_far_selected)
        
        accuracy += accuracy_score(y_test, y_pred)
        
        sensitivity += recall_score(y_test,y_pred,pos_label=1)

        specificity += recall_score(y_test,y_pred,pos_label=0)
        
        f1 += f1_score(y_test,y_pred,pos_label=1)

        balanced += balanced_accuracy(recall_score(y_test,y_pred,pos_label=1), recall_score(y_test,y_pred,pos_label=0))

        mcc += matthews_corrcoef(y_test, y_pred)
        
        far_ = calculate_FAR(y_pred_far, y_test_far, k)
        if far_ is not None:
            far += far_
            n_sz_ += 1
        
    mean_accuracy = accuracy/n_sz
    mean_specificity = specificity/n_sz
    mean_sensitivity = sensitivity/n_sz
    mean_mcc = mcc/n_sz
    mean_balanced = balanced/n_sz
    mean_f1 = f1/n_sz
    mean_far = far/n_sz_

    print(f"Mean CV F1-Score: {mean_f1:.6f}")
    print(f"Mean CV Accuracy: {mean_accuracy:.6f}")
    print(f"Mean CV Specificity: {mean_specificity:.6f}")
    print(f"Mean CV Sensitivity (Recall): {mean_sensitivity:.6f}")
    print(f"Mean CV MCC: {mean_mcc:.6f}")
    print(f"Mean CV Balanced Accuracy: {mean_balanced:.6f}")
    print(f"Mean CV FAR/h: {mean_far:.6f}")

    return mean_accuracy, mean_sensitivity, mean_specificity, mean_mcc, mean_balanced, mean_f1, mean_far
    
def save_results(scores_dict, subject_ids, file_path, mode):
    df_subject = pd.DataFrame(scores_dict, index=subject_ids)
    df_subject.index.name = "Subject"

    # Statistic DataFrame
    df_summary = pd.DataFrame({"Mean": df_subject.mean(axis=0),"STD": df_subject.std(axis=0, ddof=0)})
    df_summary.index.name = 'Metrics'

    with open(file_path, "a", encoding="utf-8") as f:
        f.write(f"Result for {mode}\n")
        df_subject.to_csv(f, index=True)
    
        f.write(f"\nSummary across subjects {mode}\n")
        df_summary.to_csv(f, index=True)

Note: "paths" in the following code is the folder containing chb files

In [ ]:
# ==========================================================================
# HANDCRAFTED FTs
# ==========================================================================

train_path = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-traditional-fts-training'
test_path = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-traditional-fts-testing'

train_paths = [os.path.join(train_path, f) for f in os.listdir(train_path)]
test_paths = [os.path.join(test_path, f, f'{f}.csv') for f in os.listdir(test_path)]
output_dir = os.path.join(save_dir, "Hancrafted")
os.makedirs(output_dir, exist_ok=True)
for k in (30,60,90):
    save_path = os.path.join(output_dir, f"k{k}.csv")
    
    for mode, model in models.items():
        if mode=='SVM':
            scaler = StandardScaler()
        else:
            scaler = None
        subject_ids = []
        scores_trad = {'Accuracy': [], 'Sensitivity': [], 'Specificity': [], 'F1': [], 'MCC': [], 'Balanced accuracy':[], 'FAR/h':[]}
        for train_path in sorted(train_paths):
            match = re.search(r'chb\d+', train_path)
            if match:
                str_chb = match.group()
                print(f'\nPROCESSING {str_chb}')
                test_path = next((path for path in test_paths if f'{str_chb}.csv' in path), None)
                if test_path is None:
                    continue
            else:
                continue
            preictal_path = os.path.join(train_path, 'preictal.csv')
            preictal_df = pd.read_csv(preictal_path)
    
            if(len(preictal_df['SeizureID'].unique()))<3:
                print('Khong du con')
                continue
            
            df_test = pd.read_csv(test_path)
    
            df_test_pivoted = pivot_df(df_test, index=['SeizureID', 'Epoch'], value_cols = df_test.drop(columns=['Band', 'SeizureID', 'Epoch']).columns.tolist())
            preictal_df['Class'] = 1
            
            interictal_path = os.path.join(train_path, 'interictal.csv')
            interictal_df = pd.read_csv(interictal_path)
            interictal_df['Class'] = 0
            
            values_cols = preictal_df.drop(columns=['SeizureID','Class', 'Epoch', 'Band']).columns.tolist()
            preictal_df = pivot_df(preictal_df, index=['SeizureID','Class', 'Epoch'], value_cols=values_cols)
            interictal_df = pivot_df(interictal_df, index=['Class', 'Epoch'], value_cols=values_cols)
            
            mean_accuracy, mean_sensitivity, mean_specificity, mean_mcc, mean_balanced, mean_f1, mean_far = Evaluate_CV(preictal_df, interictal_df, df_test_pivoted, model, k, scaler)
            if mean_accuracy is not None:
                scores_trad['Accuracy'].append(mean_accuracy)
                scores_trad['Sensitivity'].append(mean_sensitivity)
                scores_trad['Specificity'].append(mean_specificity)
                scores_trad['F1'].append(mean_f1)
                scores_trad['Balanced accuracy'].append(mean_balanced)
                scores_trad['MCC'].append(mean_mcc)
                scores_trad['FAR/h'].append(mean_far)
                subject_ids.append(str_chb)
            else:
                continue
        if len(scores_trad['Accuracy'])>0:    
            save_results(scores_trad, subject_ids, save_path, mode)

In [ ]:
# ==========================================================================
# wPLI-CONNECTIVITY FTS
# ==========================================================================

train_path = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-graph-fts0-6-training/wPLI_Graph_FTs_Training/wPLI_Features_Nodal_Thres_0.5'
test_path = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-graph-fts0-6-testing/wPLI_Thres0.6_Testing/wPLI_Features_Nodal_Thres_0.5'

train_paths = [os.path.join(train_path, f) for f in os.listdir(train_path)]
test_paths = [os.path.join(test_path, f) for f in os.listdir(test_path) if f.endswith('.csv')]
output_dir = os.path.join(save_dir, "wPLI")
os.makedirs(output_dir, exist_ok=True)
for k in (30,60,90):
    save_path = os.path.join(output_dir, f"k{k}.csv")
    for mode, model in models.items():
        if mode=='SVM':
            scaler = StandardScaler()
        else:
            scaler = None
        subject_ids = []
        scores_wpli = {'Accuracy': [], 'Sensitivity': [], 'Specificity': [], 'F1': [], 'MCC': [], 'Balanced accuracy':[], 'FAR/h':[]}
        for train_path in sorted(train_paths):
            match = re.search(r'chb\d+', train_path)
            if match:
                str_chb = match.group()
                print(f'\nPROCESSING {str_chb}')
                test_path = next((path for path in test_paths if f'{str_chb}.csv' in path), None)
                if test_path is None:
                    continue
            else:
                continue
            preictal_path = os.path.join(train_path, 'preictal.csv')
            preictal_df = pd.read_csv(preictal_path)
    
            if(len(preictal_df['SeizureID'].unique()))<3:
                print('Khong du con')
                continue
            
            df_test = pd.read_csv(test_path)
            df_test = remove_duplicate_columns(df_test)
            df_test_pivoted = pivot_df(df_test, index=['SeizureID', 'Epoch'], value_cols = df_test.drop(columns=['Band', 'SeizureID', 'Epoch']).columns.tolist())
            preictal_df['Class'] = 1
            
            interictal_path = os.path.join(train_path, 'interictal.csv')
            interictal_df = pd.read_csv(interictal_path)
            interictal_df['Class'] = 0
    
            preictal_df = remove_duplicate_columns(preictal_df)
            interictal_df = remove_duplicate_columns(interictal_df)
            
            values_cols = preictal_df.drop(columns=['SeizureID','Class', 'Epoch', 'Band']).columns.tolist()
            preictal_df = pivot_df(preictal_df, index=['SeizureID','Class', 'Epoch'], value_cols=values_cols)
            interictal_df = pivot_df(interictal_df, index=['Class', 'Epoch'], value_cols=values_cols)
            
            mean_accuracy, mean_sensitivity, mean_specificity, mean_mcc, mean_balanced, mean_f1, mean_far = Evaluate_CV(preictal_df, interictal_df, df_test_pivoted, model,k, scaler)
            if mean_accuracy is not None:
                scores_wpli['Accuracy'].append(mean_accuracy)
                scores_wpli['Sensitivity'].append(mean_sensitivity)
                scores_wpli['Specificity'].append(mean_specificity)
                scores_wpli['F1'].append(mean_f1)
                scores_wpli['Balanced accuracy'].append(mean_balanced)
                scores_wpli['MCC'].append(mean_mcc)
                scores_wpli['FAR/h'].append(mean_far)
                subject_ids.append(str_chb)
            else:
                continue
        if len(scores_wpli['Accuracy'])>0:    
            save_results(scores_wpli, subject_ids, save_path, mode)'''

In [ ]:
# ==========================================================================
# PLI-CONNECTIVITY FTS
# ==========================================================================

train_path = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-graph-fts0-6-training/PLI_Graph_FTs_Training/PLI_Features_Nodal_Thres_0.5'
test_path = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-graph-fts0-6-testing/PLI_Thres0.6_Testing/PLI_Features_Nodal_Thres_0.5'

train_paths = [os.path.join(train_path, f) for f in os.listdir(train_path)]
test_paths = [os.path.join(test_path, f) for f in os.listdir(test_path) if f.endswith('.csv')]

output_dir = os.path.join(save_dir, "PLI")
os.makedirs(output_dir, exist_ok=True)
for k in (30,60,90):
    save_path = os.path.join(output_dir, f"k{k}.csv")

    for mode, model in models.items():
        if mode=='SVM':
            scaler = StandardScaler()
        else:
            scaler = None
        subject_ids = []
        scores_pli = {'Accuracy': [], 'Sensitivity': [], 'Specificity': [], 'F1': [], 'MCC': [], 'Balanced accuracy':[], 'FAR/h':[]}
        for train_path in sorted(train_paths):
            match = re.search(r'chb\d+', train_path)
            if match:
                str_chb = match.group()
                print(f'\nPROCESSING {str_chb}')
                test_path = next((path for path in test_paths if f'{str_chb}.csv' in path), None)
                if test_path is None:
                    continue
            else:
                continue
            preictal_path = os.path.join(train_path, 'preictal.csv')
            preictal_df = pd.read_csv(preictal_path)
    
            if(len(preictal_df['SeizureID'].unique()))<3:
                print('Khong du con')
                continue
            df_test = pd.read_csv(test_path)
            df_test = remove_duplicate_columns(df_test)
            df_test_pivoted = pivot_df(df_test, index=['SeizureID', 'Epoch'], value_cols = df_test.drop(columns=['Band', 'SeizureID', 'Epoch']).columns.tolist())
            
            preictal_df['Class'] = 1
            
            interictal_path = os.path.join(train_path, 'interictal.csv')
            interictal_df = pd.read_csv(interictal_path)
            interictal_df['Class'] = 0
    
            preictal_df = remove_duplicate_columns(preictal_df)
            interictal_df = remove_duplicate_columns(interictal_df)
            
            values_cols = preictal_df.drop(columns=['SeizureID','Class', 'Epoch', 'Band']).columns.tolist()
            preictal_df = pivot_df(preictal_df, index=['SeizureID','Class', 'Epoch'], value_cols=values_cols)
            interictal_df = pivot_df(interictal_df, index=['Class', 'Epoch'], value_cols=values_cols)
            
            mean_accuracy, mean_sensitivity, mean_specificity, mean_mcc, mean_balanced, mean_f1, mean_far = Evaluate_CV(preictal_df, interictal_df, df_test_pivoted, model,k, scaler)
            if mean_accuracy is not None:
                scores_pli['Accuracy'].append(mean_accuracy)
                scores_pli['Sensitivity'].append(mean_sensitivity)
                scores_pli['Specificity'].append(mean_specificity)
                scores_pli['F1'].append(mean_f1)
                scores_pli['Balanced accuracy'].append(mean_balanced)
                scores_pli['MCC'].append(mean_mcc)
                scores_pli['FAR/h'].append(mean_far)
                subject_ids.append(str_chb)
            else:
                continue
        if len(scores_pli['Accuracy'])>0:    
            save_results(scores_pli, subject_ids, save_path, mode)

In [ ]:
# ==========================================================================
# Hybrid-PLI FTS
# ==========================================================================

pli_train = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-graph-fts0-6-training/PLI_Graph_FTs_Training/PLI_Features_Nodal_Thres_0.5'
pli_test = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-graph-fts0-6-testing/PLI_Thres0.6_Testing/PLI_Features_Nodal_Thres_0.5'
pli_train_paths = [os.path.join(pli_train, f) for f in os.listdir(pli_train)]
pli_test_paths = [os.path.join(pli_test, f) for f in os.listdir(pli_test) if f.endswith('.csv')]

trad_train = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-traditional-fts-training'
trad_test = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-traditional-fts-testing'
trad_train_paths = [os.path.join(trad_train, f) for f in os.listdir(trad_train)]
trad_test_paths = [os.path.join(trad_test, f, f'{f}.csv') for f in os.listdir(trad_test)]

output_dir = os.path.join(save_dir, "PLI-Handcrafted")
os.makedirs(output_dir, exist_ok=True)
for k in (30,60,90):
    save_path = os.path.join(output_dir, f"k{k}.csv")
    for mode, model in models.items():
        if mode=='SVM':
            scaler = StandardScaler()
        else:
            scaler = None
        subject_ids = []
        scores_pli_trad = {'Accuracy': [], 'Sensitivity': [], 'Specificity': [], 'F1': [], 'MCC': [], 'Balanced accuracy':[], 'FAR/h':[]}
        for pli_train_path in sorted(pli_train_paths):
            match = re.search(r'chb\d+', pli_train_path)
            if match:
                str_chb = match.group()
                print(f'\nPROCESSING {str_chb}')
                pli_test_path = next((path for path in pli_test_paths if f'{str_chb}.csv' in path), None)
                trad_train_path = next((path for path in trad_train_paths if str_chb in path), None)
                trad_test_path = next((path for path in trad_test_paths if f'{str_chb}.csv' in path), None)
                if (pli_test_path is None) or (trad_train_path is None) or (trad_test_path is None):
                    continue
            else:
                continue
            preictal_path_pli = os.path.join(pli_train_path, 'preictal.csv')
            preictal_path_trad = os.path.join(trad_train_path, 'preictal.csv')
            
            preictal_df_pli = pd.read_csv(preictal_path_pli)
            preictal_df_trad = pd.read_csv(preictal_path_trad)
    
            if(len(preictal_df_pli['SeizureID'].unique()))<3:
                print('Khong du con')
                continue
            df_test_pli = pd.read_csv(pli_test_path)
            
            df_test_pli = remove_duplicate_columns(df_test_pli)
            df_test_trad = pd.read_csv(trad_test_path)
    
            df_test_pli_pivot = pivot_df(df_test_pli, index=['SeizureID', 'Epoch'], value_cols = df_test_pli.drop(columns=['Band', 'SeizureID', 'Epoch']).columns.tolist())
            df_test_trad_pivot = pivot_df(df_test_trad, index=['SeizureID', 'Epoch'], value_cols = df_test_trad.drop(columns=['Band', 'SeizureID', 'Epoch']).columns.tolist())
            df_test_pivoted = pd.merge(df_test_pli_pivot, df_test_trad_pivot, on = ['SeizureID','Epoch'])
            
            preictal_df_pli['Class'] = 1
            preictal_df_trad['Class'] = 1
            
            interictal_path_pli = os.path.join(pli_train_path, 'interictal.csv')
            interictal_df_pli = pd.read_csv(interictal_path_pli)
            interictal_df_pli['Class'] = 0
    
            interictal_path_trad = os.path.join(trad_train_path, 'interictal.csv')
            interictal_df_trad = pd.read_csv(interictal_path_trad)
            interictal_df_trad['Class'] = 0
    
            preictal_df_pli = remove_duplicate_columns(preictal_df_pli)
            interictal_df_pli = remove_duplicate_columns(interictal_df_pli)
            
            values_cols_pli = preictal_df_pli.drop(columns=['SeizureID', 'Class', 'Epoch', 'Band']).columns.tolist()
            values_cols_trad = preictal_df_trad.drop(columns=['SeizureID', 'Class', 'Epoch', 'Band']).columns.tolist()
    
            preictal_df_pli = pivot_df(preictal_df_pli, index=['SeizureID', 'Class', 'Epoch'], value_cols = values_cols_pli)
            preictal_df_trad = pivot_df(preictal_df_trad, index=['SeizureID','Class', 'Epoch'], value_cols = values_cols_trad)
    
            interictal_df_pli = pivot_df(interictal_df_pli, index=['Class', 'Epoch'], value_cols = values_cols_pli)
            interictal_df_trad = pivot_df(interictal_df_trad, index=['Class', 'Epoch'], value_cols = values_cols_trad)
    
            preictal_df = pd.merge(preictal_df_pli, preictal_df_trad, on = ['SeizureID', 'Class', 'Epoch'])
            interictal_df = pd.merge(interictal_df_pli, interictal_df_trad, on = ['Class', 'Epoch'])
            
            mean_accuracy, mean_sensitivity, mean_specificity, mean_mcc, mean_balanced, mean_f1, mean_far = Evaluate_CV(preictal_df, interictal_df, df_test_pivoted, model, k, scaler)
            
            if mean_accuracy is not None:
                scores_pli_trad['Accuracy'].append(mean_accuracy)
                scores_pli_trad['Sensitivity'].append(mean_sensitivity)
                scores_pli_trad['Specificity'].append(mean_specificity)
                scores_pli_trad['F1'].append(mean_f1)
                scores_pli_trad['Balanced accuracy'].append(mean_balanced)
                scores_pli_trad['MCC'].append(mean_mcc)
                scores_pli_trad['FAR/h'].append(mean_far)
                subject_ids.append(str_chb)
            else:
                continue
        if len(scores_pli_trad['Accuracy'])>0:    
            save_results(scores_pli_trad, subject_ids, save_path, mode)

In [ ]:
# ==========================================================================
# Hybrid-wPLI FTS
# ==========================================================================

pli_train = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-graph-fts0-6-training/wPLI_Graph_FTs_Training/wPLI_Features_Nodal_Thres_0.5'
pli_test = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-graph-fts0-6-testing/wPLI_Thres0.6_Testing/wPLI_Features_Nodal_Thres_0.5'
pli_train_paths = [os.path.join(pli_train, f) for f in os.listdir(pli_train)]
pli_test_paths = [os.path.join(pli_test, f) for f in os.listdir(pli_test) if f.endswith('.csv')]
pli_paths = sorted(zip(pli_train_paths, pli_test_paths))

trad_train = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-traditional-fts-training'
trad_test = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-traditional-fts-testing'
trad_train_paths = [os.path.join(trad_train, f) for f in os.listdir(trad_train)]
trad_test_paths = [os.path.join(trad_test, f, f'{f}.csv') for f in os.listdir(trad_test)]
trad_paths = sorted(zip(trad_train_paths, trad_test_paths))

output_dir = os.path.join(save_dir, "wPLI-Handcrafted")
os.makedirs(output_dir, exist_ok=True)
for k in (30,60,90):
    save_path = os.path.join(output_dir, f"k{k}.csv")
    for mode, model in models.items():
        if mode=='SVM':
            scaler = StandardScaler()
        else:
            scaler = None
        subject_ids = []
        scores_wpli_trad = {'Accuracy': [], 'Sensitivity': [], 'Specificity': [], 'F1': [], 'MCC': [], 'Balanced accuracy':[], 'FAR/h':[]}
        for pli_train_path in sorted(pli_train_paths):
            match = re.search(r'chb\d+', pli_train_path)
            if match:
                str_chb = match.group()
                print(f'\nPROCESSING {str_chb}')
                pli_test_path = next((path for path in pli_test_paths if f'{str_chb}.csv' in path), None)
                trad_train_path = next((path for path in trad_train_paths if str_chb in path), None)
                trad_test_path = next((path for path in trad_test_paths if f'{str_chb}.csv' in path), None)
                if pli_test_path is None or trad_train_path is None or trad_test_path is None:
                    continue
            else:
                continue
            preictal_path_pli = os.path.join(pli_train_path, 'preictal.csv')
            preictal_path_trad = os.path.join(trad_train_path, 'preictal.csv')
            
            preictal_df_pli = pd.read_csv(preictal_path_pli)
            preictal_df_trad = pd.read_csv(preictal_path_trad)
    
            if(len(preictal_df_pli['SeizureID'].unique()))<3:
                print('Khong du con')
                continue
            df_test_pli = pd.read_csv(pli_test_path)
            
            df_test_pli = remove_duplicate_columns(df_test_pli)
            df_test_trad = pd.read_csv(trad_test_path)
    
            df_test_pli_pivot = pivot_df(df_test_pli, index=['SeizureID', 'Epoch'], value_cols = df_test_pli.drop(columns=['Band', 'SeizureID', 'Epoch']).columns.tolist())
            df_test_trad_pivot = pivot_df(df_test_trad, index=['SeizureID', 'Epoch'], value_cols = df_test_trad.drop(columns=['Band', 'SeizureID', 'Epoch']).columns.tolist())
            df_test_pivoted = pd.merge(df_test_pli_pivot, df_test_trad_pivot, on = ['SeizureID','Epoch'])
            
            preictal_df_pli['Class'] = 1
            preictal_df_trad['Class'] = 1
            
            interictal_path_pli = os.path.join(pli_train_path, 'interictal.csv')
            interictal_df_pli = pd.read_csv(interictal_path_pli)
            interictal_df_pli['Class'] = 0
    
            interictal_path_trad = os.path.join(trad_train_path, 'interictal.csv')
            interictal_df_trad = pd.read_csv(interictal_path_trad)
            interictal_df_trad['Class'] = 0
    
            preictal_df_pli = remove_duplicate_columns(preictal_df_pli)
            interictal_df_pli = remove_duplicate_columns(interictal_df_pli)
            
            values_cols_pli = preictal_df_pli.drop(columns=['SeizureID', 'Class', 'Epoch', 'Band']).columns.tolist()
            values_cols_trad = preictal_df_trad.drop(columns=['SeizureID', 'Class', 'Epoch', 'Band']).columns.tolist()
    
            preictal_df_pli = pivot_df(preictal_df_pli, index=['SeizureID', 'Class', 'Epoch'], value_cols = values_cols_pli)
            preictal_df_trad = pivot_df(preictal_df_trad, index=['SeizureID','Class', 'Epoch'], value_cols = values_cols_trad)
    
            interictal_df_pli = pivot_df(interictal_df_pli, index=['Class', 'Epoch'], value_cols = values_cols_pli)
            interictal_df_trad = pivot_df(interictal_df_trad, index=['Class', 'Epoch'], value_cols = values_cols_trad)
    
            preictal_df = pd.merge(preictal_df_pli, preictal_df_trad, on = ['SeizureID', 'Class', 'Epoch'])
            interictal_df = pd.merge(interictal_df_pli, interictal_df_trad, on = ['Class', 'Epoch'])
            
            mean_accuracy, mean_sensitivity, mean_specificity, mean_mcc, mean_balanced, mean_f1, mean_far = Evaluate_CV(preictal_df, interictal_df, df_test_pivoted, model, k,scaler)
            
            if mean_accuracy is not None:
                scores_wpli_trad['Accuracy'].append(mean_accuracy)
                scores_wpli_trad['Sensitivity'].append(mean_sensitivity)
                scores_wpli_trad['Specificity'].append(mean_specificity)
                scores_wpli_trad['F1'].append(mean_f1)
                scores_wpli_trad['Balanced accuracy'].append(mean_balanced)
                scores_wpli_trad['MCC'].append(mean_mcc)
                scores_wpli_trad['FAR/h'].append(mean_far)
                subject_ids.append(str_chb)
            else:
                continue
        if len(scores_wpli_trad['Accuracy'])>0:    
            save_results(scores_wpli_trad, subject_ids, save_path, mode)